# 05 — Failure Prediction Modeling

Trains and compares Logistic Regression, Random Forest, Gradient Boosting, Extra Trees, and XGBoost classifiers to predict `Failure_Within_30_Days`. Selects a final model favoring **Recall** (minimizing missed failures), and saves it with Joblib.

This notebook wraps the logic in `src/train_failure_model.py` — run that script directly for a clean, reproducible pipeline run.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..') / 'src'))

import train_failure_model

train_failure_model.main()

2026-09-25 11:01:12,739 | INFO | Building training dataframe (clean + feature engineer)...


2026-09-25 11:01:13,131 | INFO | Starting data cleaning: 22440 rows, 26 columns


2026-09-25 11:01:13,172 | INFO |   Removed 440 exact duplicate rows


2026-09-25 11:01:13,174 | INFO |   Temperature_C: flagging 132 out-of-range readings as missing


2026-09-25 11:01:13,176 | INFO |   Pressure_Bar: flagging 197 out-of-range readings as missing


...

2026-09-25 11:01:59,123 | INFO |   XGBoost                Acc=0.590 Prec=0.497 Recall=0.508 F1=0.503 ROC-AUC=0.605  FN=882


2026-09-25 11:01:59,128 | INFO | Saved model comparison -> /home/claude/oil_gas_predictive_maintenance/reports/model_performance.csv


2026-09-25 11:01:59,133 | INFO | Selected final model: Logistic Regression


2026-09-25 11:01:59,137 | INFO | Saved model + scaler + encoder + feature_names to /home/claude/oil_gas_predictive_maintenance/models



              Model  Accuracy  Precision  Recall  F1_Score  ROC_AUC  False_Negatives  False_Positives
Logistic Regression    0.5957     0.5036  0.5792    0.5388   0.6310              755             1024
            XGBoost    0.5900     0.4973  0.5084    0.5028   0.6049              882              922
        Extra Trees    0.5902     0.4974  0.4799    0.4885   0.6161              933              870
      Random Forest    0.6000     0.5106  0.4565    0.4820   0.6213              975              785
  Gradient Boosting    0.6082     0.5348  0.2999    0.3843   0.6155             1256              468



## Review saved model comparison report

In [2]:
import pandas as pd
pd.read_csv('../reports/model_performance.csv')

,Model,Accuracy,Precision,Recall,F1_Score,ROC_AUC,False_Negatives,False_Positives
0,Logistic Regression,0.5957,0.5036,0.5792,0.5388,0.6310,755,1024
1,XGBoost,0.5900,0.4973,0.5084,0.5028,0.6049,882,922
2,Extra Trees,0.5902,0.4974,0.4799,0.4885,0.6161,933,870
3,Random Forest,0.6000,0.5106,0.4565,0.4820,0.6213,975,785
4,Gradient Boosting,0.6082,0.5348,0.2999,0.3843,0.6155,1256,468


## Review feature importance (explainability)

In [3]:
import plotly.express as px
import plotly.io as pio
pio.templates.default = 'plotly_dark'

imp = pd.read_csv('../reports/feature_importance.csv', index_col=0).head(15)
px.bar(imp, orientation='h', title='Top 15 Feature Importances — Failure Prediction Model')

## Confusion matrix for the selected model

In [4]:
import joblib
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

model = joblib.load('../models/failure_prediction_model.pkl')
print('Loaded model:', type(model).__name__)

Loaded model: LogisticRegression
